<a href="https://colab.research.google.com/github/PapBill/Apache-Spark-Implementations/blob/main/Spark_Implementations.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

---

# **Spark Setup**

---

In [3]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
!pip3 install pyspark
from pyspark import SparkContext, SparkConf
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *

import re

#Initializing Spark
setup = SparkConf().setAppName("Asignment2").setMaster("local[*]")
sc = SparkContext.getOrCreate(setup)
Tweets_sc = SparkSession.builder.getOrCreate()
DNA_sc = sc
SherlockHolmes_sc = sc


----
---

# **SherlockHolmes.txt Acronym Counter**
---
--------

In [4]:
#Reading & loading data to Spark
rdd1 = SherlockHolmes_sc.textFile("/content/drive/MyDrive/Colab_Notebooks/SherlockHolmes.txt")

In [5]:
words = rdd1.map(lambda line : line.lower()) \
           .flatMap(lambda line : line.split())

In [6]:
words.takeSample(False, 25, seed=110)

['severity',
 'punished',
 'astonishment,',
 'he',
 '"has',
 'side',
 '304',
 'be',
 'collapse,',
 'began',
 'not',
 'he',
 'basis,',
 'just',
 'from',
 'go',
 'he',
 'disease--_antitoxic',
 'found',
 'of',
 'the',
 'example,',
 'drawing',
 '(593-594).',
 'do']

In [7]:
# Email Tokenization
email_expr = r"[a-zA-Z0-9.-_]+@[a-zA-Z0-9.-_]+\.[a-zA-Z]{2,}"

def emailFun(txt):

  def emailSplit(mail):
# Getting matched email string "localname@domain.TLD.TLD" -> locanalme + domain + TLD + TLD
      mail = mail.group(0)
      localname = mail.split("@")[0]
      domainTLD = mail.split("@")[1]
      domain =domainTLD.split(".")[0]
# 2 TLD case
      if len(domainTLD.split(".")) == 3 :
        tld1 = domainTLD.split(".")[1]
        tld2 = domainTLD.split(".")[2]
        return f"{localname} {domain} {tld1} {tld2}"
# 1 TLD case
      else :
        tld = domainTLD.split(".")[1]
        return f"{localname} {domain} {tld}"

  return re.sub(email_expr, emailSplit, txt)

#Fistly, we tokenize email cases before removing punctuation marks
words = words.map(emailFun)

In [8]:
#Punctuation marks handling

def Punctuation(words) :
  words = re.sub(r"[!#$%;^&*)(\[\]{}\\:'\"<>/?,_|=+]", "", re.sub(r"[-]", " ", words))
  return words

#Acronyms handling

def fullstop(words) :
  if re.search(r"\.+", words) :
    return re.sub(r"\.+", "", words)
  else :
    return re.sub(r"\.-"," ", words)

words = words.map(Punctuation).map(fullstop)
print(words.takeSample(False, 30 , seed=100))
#There some cases with more than 1 element : 'son in law'

['is', 'fee', 'from', 'carried', 'not', 'several', 'to', 'benefit', 'them', 'centre', 'ceases', 'to', 'acid', 'consciousness', 'retired', 'own', 'son in law', 'he', 'chapter', 'provision', 'throbbing', 'offended', 'south', 'natasha', 'cheeks', 'patient', 'and', 'how', 'passed', 'soft']


In [9]:
# Removing numbers & removing strings empty elemets & removing words with less than 4 letters.
words = words.map(lambda word : re.sub(r"[0-9]", "", word)) \
             .flatMap(lambda line : line.split()) \
             .filter(lambda word : word.strip() != "") \
             .filter(lambda words : len(words) > 3)
print(words.take(20))
#print(words.takeSample(False, 30, seed=110))

['project', 'gutenberg', 'ebook', 'adventures', 'sherlock', 'holmes', 'arthur', 'conan', 'doyle', 'series', 'arthur', 'conan', 'doyle', 'copyright', 'laws', 'changing', 'over', 'world', 'sure', 'check']


In [10]:
# Converting into numeronyms
def numeronyms(word) :
  word = word[0] + str(len(word)-2) + word[-1]
  return word

original_words = words
num_words = words.map(numeronyms)
print(f"Orinigal words :\n{original_words.take(10)}\n")
print("Acronyms of the original words : ")
print(num_words.take(10))


Orinigal words :
['project', 'gutenberg', 'ebook', 'adventures', 'sherlock', 'holmes', 'arthur', 'conan', 'doyle', 'series']

Acronyms of the original words : 
['p5t', 'g7g', 'e3k', 'a8s', 's6k', 'h4s', 'a4r', 'c3n', 'd3e', 's4s']


In [11]:
# Mapping acronyms into : (key,1)
Mapped_num_words = num_words.map(lambda x : (x,1))
Mapped_num_words.take(10)

[('p5t', 1),
 ('g7g', 1),
 ('e3k', 1),
 ('a8s', 1),
 ('s6k', 1),
 ('h4s', 1),
 ('a4r', 1),
 ('c3n', 1),
 ('d3e', 1),
 ('s4s', 1)]

In [12]:
# Reducing by key & sorting by value : (key, value)
Reduced_num_words = Mapped_num_words.reduceByKey(lambda x,y : x+y).sortBy(lambda x : x[1], ascending = False)
print(f"Final sorted result :\n{Reduced_num_words.take(5)}")
print(f"\nTotal number of reduced acronyms of  book 'SherlockHolmes.txt': {Reduced_num_words.count()}")

# Keeping acronyms that appeared more than 10 times.
final_result = Reduced_num_words.filter(lambda value : value[1] >=10)

Final sorted result :
[('t2t', 12390), ('w2h', 9995), ('t3e', 6626), ('f2m', 6409), ('p4e', 5046)]

Total number of reduced acronyms of  book 'SherlockHolmes.txt': 3376


In [13]:
# Saving to Google Drive...
print(f"Final counted acronyms of  book 'SherlockHolmes.txt': {final_result.count()}")
#final_result.saveAsTextFile("/content/drive/MyDrive/Colab_Notebooks/sherlock_Numeronyms")

Final counted acronyms of  book 'SherlockHolmes.txt': 2158


---
---
# **SherlockHolmes.txt Average Word Length**
---
---

In [40]:
#Reading & loading data to Spark
rdd2 = SherlockHolmes_sc.textFile("/content/drive/MyDrive/Colab_Notebooks/SherlockHolmes.txt")

In [41]:
words = rdd2.map(lambda line : line.lower()) \
           .flatMap(lambda line : line.split())
words.takeSample(False, 25, seed=110)

['severity',
 'punished',
 'astonishment,',
 'he',
 '"has',
 'side',
 '304',
 'be',
 'collapse,',
 'began',
 'not',
 'he',
 'basis,',
 'just',
 'from',
 'go',
 'he',
 'disease--_antitoxic',
 'found',
 'of',
 'the',
 'example,',
 'drawing',
 '(593-594).',
 'do']

In [42]:
# Email Tokenization
email_expr = r"[a-zA-Z0-9.-_]+@[a-zA-Z0-9.-_]+\.[a-zA-Z]{2,}"

def emailFun(txt):

  def emailSplit(mail):
# Getting matched email string "localname@domain.TLD.TLD"
      mail = mail.group(0)
      localname = mail.split("@")[0]
      domainTLD = mail.split("@")[1]
      domain =domainTLD.split(".")[0]
# 2 TLD case
      if len(domainTLD.split(".")) == 3 :
        tld1 = domainTLD.split(".")[1]
        tld2 = domainTLD.split(".")[2]
        return f"{localname} {domain} {tld1} {tld2}"
# 1 TLD case
      else :
        tld = domainTLD.split(".")[1]
        return f"{localname} {domain} {tld}"

  return re.sub(email_expr, emailSplit, txt)

#Fistly, we tokenize email cases before removing punctuation marks
words = words.map(emailFun)

In [43]:
#Punctuation marks handling

def Punctuation(words) :
  words = re.sub(r"[!#$%;~^&*)(\[\]{}\\:'\"<>/?,_|=+]", "", re.sub(r"[-]", " ", words))
  return words

#Acronyms handling

def fullstop(words) :
  if re.search(r"\.+", words) :
    return re.sub(r"\.+", "", words)
  else :
    return re.sub(r"\.-"," ", words)

words = words.map(Punctuation).map(fullstop)
print(words.takeSample(False, 30 , seed=100))

['is', 'fee', 'from', 'carried', 'not', 'several', 'to', 'benefit', 'them', 'centre', 'ceases', 'to', 'acid', 'consciousness', 'retired', 'own', 'son in law', 'he', 'chapter', 'provision', 'throbbing', 'offended', 'south', 'natasha', 'cheeks', 'patient', 'and', 'how', 'passed', 'soft']


In [44]:
# Removing numbers & removing strings empty elemets.
words = words.map(lambda word : re.sub(r"[0-9]", "", word)) \
             .flatMap(lambda line : line.split()) \
             .filter(lambda word : word.strip() != "")
print(words.take(20))

['the', 'project', 'gutenberg', 'ebook', 'of', 'the', 'adventures', 'of', 'sherlock', 'holmes', 'by', 'sir', 'arthur', 'conan', 'doyle', 'in', 'our', 'series', 'by', 'sir']


In [45]:
# Removing numbers & removing strings empty elemets.
words = words.map(lambda word : re.sub(r"[0-9]", "", word)) \
             .flatMap(lambda line : line.split()) \
             .filter(lambda word : word.strip() != "")
print(words.take(20))

['the', 'project', 'gutenberg', 'ebook', 'of', 'the', 'adventures', 'of', 'sherlock', 'holmes', 'by', 'sir', 'arthur', 'conan', 'doyle', 'in', 'our', 'series', 'by', 'sir']


In [46]:
# Converting into numeronyms
def numeronyms(word) :
  word = word[0] + str(len(word))
  return word

original_words = words
num_words = words.map(numeronyms)
print(f"Orinigal words :\n{original_words.take(10)}\n")
print("Acronyms of the original words : ")
print(num_words.take(10))


Orinigal words :
['the', 'project', 'gutenberg', 'ebook', 'of', 'the', 'adventures', 'of', 'sherlock', 'holmes']

Acronyms of the original words : 
['t3', 'p7', 'g9', 'e5', 'o2', 't3', 'a10', 'o2', 's8', 'h6']


In [47]:
# Mapping numeronyms into : (key,lenth)
Mapped_words = original_words.map(lambda x : (x[0],len(x)))
Mapped_words.take(5)

[('t', 3), ('p', 7), ('g', 9), ('e', 5), ('o', 2)]

In [48]:
# Mapping numeronyms into : (key, (lenth, 1))
Mapped_words = Mapped_words.map(lambda x : (x[0], (x[1],1)))
Mapped_words.take(5)

[('t', (3, 1)), ('p', (7, 1)), ('g', (9, 1)), ('e', (5, 1)), ('o', (2, 1))]

In [49]:
# Reducing numeronyms into : (key (sum, count))
letter_sum_count = Mapped_words.reduceByKey(lambda x,y : (x[0]+y[0], x[1]+y[1]))
letter_sum_count.take(5)

[('t', (631582, 174808)),
 ('p', (273955, 39484)),
 ('g', (105231, 18114)),
 ('o', (232993, 77764)),
 ('b', (218728, 48778))]

In [55]:
# Calculating the average length of words beginning with a specific character [a-z]
letter_avg = letter_sum_count.map(lambda x : (x[0],(x[1][0]/x[1][1]))).sortBy(lambda x : x[1], ascending = False)
letter_avg.collect()

[('c', 7.108178679326715),
 ('e', 7.049772837903618),
 ('q', 6.970935513169845),
 ('p', 6.938380103332996),
 ('r', 6.7784986425020275),
 ('d', 6.295982364629905),
 ('v', 5.890387444825895),
 ('g', 5.809373964889036),
 ('z', 5.7238493723849375),
 ('s', 5.718596444503628),
 ('u', 5.577326085243835),
 ('j', 5.538819875776397),
 ('k', 5.291472063656807),
 ('l', 5.264508534432019),
 ('f', 5.121891999159526),
 ('m', 5.092711833970395),
 ('n', 4.748726454211586),
 ('b', 4.484152691787281),
 ('w', 4.255282487113214),
 ('h', 3.7865168539325844),
 ('y', 3.715472481827622),
 ('a', 3.7125107344283097),
 ('t', 3.6130039815111434),
 ('i', 3.4383220364932012),
 ('o', 2.9961550331773057),
 ('x', 2.6493212669683257)]

In [ ]:
# Saving to Google Drive...
#letter_avg.saveAsTextFile("/content/drive/MyDrive/Colab_Notebooks/sherlock_avg_lenth_of_words")

---
---
#**DNA k-mers**
---
---

In [56]:
rdd2 = DNA_sc.textFile("/content/drive/MyDrive/Colab_Notebooks/ecoli.txt")
rdd2.take(5)

['AGCTTTTCATTCTGACTGCAACGGGCAATATGTCTCTGTGTGGATTAAAAAAAGAGTGTCTGATAGCAGC',
 'TTCTGAACTGGTTACCTGCCGTGAGTAAATTAAAATTTTATTGACTTAGGTCACTAAATACTTTAACCAA',
 'TATAGGCATAGCGCACAGACAGATAAAAATTACAGAGTACACAACATCCATGAAACGCATTAGCACCACC',
 'ATTACCACCACCATCACCATTACCACAGGTAACGGTGCGGGCTGACGCGTACAGGAAACACAGAAAAAAG',
 'CCCGCACCTGACAGTGCGGGCTTTTTTTTTCGACCAAAGGTAACGAGGTAACAACCATGCGAGTGTTGAA']

In [57]:
rdd2.map(lambda x : x.split('/n')).take(5)

[['AGCTTTTCATTCTGACTGCAACGGGCAATATGTCTCTGTGTGGATTAAAAAAAGAGTGTCTGATAGCAGC'],
 ['TTCTGAACTGGTTACCTGCCGTGAGTAAATTAAAATTTTATTGACTTAGGTCACTAAATACTTTAACCAA'],
 ['TATAGGCATAGCGCACAGACAGATAAAAATTACAGAGTACACAACATCCATGAAACGCATTAGCACCACC'],
 ['ATTACCACCACCATCACCATTACCACAGGTAACGGTGCGGGCTGACGCGTACAGGAAACACAGAAAAAAG'],
 ['CCCGCACCTGACAGTGCGGGCTTTTTTTTTCGACCAAAGGTAACGAGGTAACAACCATGCGAGTGTTGAA']]

In [58]:
#Splitting given string into k-mers
def splitter(line) :
  pairs = []

  for mers in range(2,5) :
    for i in range(0,len(line)-mers+1) :
     pairs.append(''.join(line[i:i+mers]))

  return pairs

pairs = rdd2.map(lambda line : splitter(line))
pairs.take(1)

[['AG',
  'GC',
  'CT',
  'TT',
  'TT',
  'TT',
  'TC',
  'CA',
  'AT',
  'TT',
  'TC',
  'CT',
  'TG',
  'GA',
  'AC',
  'CT',
  'TG',
  'GC',
  'CA',
  'AA',
  'AC',
  'CG',
  'GG',
  'GG',
  'GC',
  'CA',
  'AA',
  'AT',
  'TA',
  'AT',
  'TG',
  'GT',
  'TC',
  'CT',
  'TC',
  'CT',
  'TG',
  'GT',
  'TG',
  'GT',
  'TG',
  'GG',
  'GA',
  'AT',
  'TT',
  'TA',
  'AA',
  'AA',
  'AA',
  'AA',
  'AA',
  'AA',
  'AG',
  'GA',
  'AG',
  'GT',
  'TG',
  'GT',
  'TC',
  'CT',
  'TG',
  'GA',
  'AT',
  'TA',
  'AG',
  'GC',
  'CA',
  'AG',
  'GC',
  'AGC',
  'GCT',
  'CTT',
  'TTT',
  'TTT',
  'TTC',
  'TCA',
  'CAT',
  'ATT',
  'TTC',
  'TCT',
  'CTG',
  'TGA',
  'GAC',
  'ACT',
  'CTG',
  'TGC',
  'GCA',
  'CAA',
  'AAC',
  'ACG',
  'CGG',
  'GGG',
  'GGC',
  'GCA',
  'CAA',
  'AAT',
  'ATA',
  'TAT',
  'ATG',
  'TGT',
  'GTC',
  'TCT',
  'CTC',
  'TCT',
  'CTG',
  'TGT',
  'GTG',
  'TGT',
  'GTG',
  'TGG',
  'GGA',
  'GAT',
  'ATT',
  'TTA',
  'TAA',
  'AAA',
  'AAA',
  'AAA',
  'AAA'

In [59]:
#Converting every line into : [(line),1]
mapped_pairs = pairs.map(lambda x : (x,1))
mapped_pairs.take(1)

[(['AG',
   'GC',
   'CT',
   'TT',
   'TT',
   'TT',
   'TC',
   'CA',
   'AT',
   'TT',
   'TC',
   'CT',
   'TG',
   'GA',
   'AC',
   'CT',
   'TG',
   'GC',
   'CA',
   'AA',
   'AC',
   'CG',
   'GG',
   'GG',
   'GC',
   'CA',
   'AA',
   'AT',
   'TA',
   'AT',
   'TG',
   'GT',
   'TC',
   'CT',
   'TC',
   'CT',
   'TG',
   'GT',
   'TG',
   'GT',
   'TG',
   'GG',
   'GA',
   'AT',
   'TT',
   'TA',
   'AA',
   'AA',
   'AA',
   'AA',
   'AA',
   'AA',
   'AG',
   'GA',
   'AG',
   'GT',
   'TG',
   'GT',
   'TC',
   'CT',
   'TG',
   'GA',
   'AT',
   'TA',
   'AG',
   'GC',
   'CA',
   'AG',
   'GC',
   'AGC',
   'GCT',
   'CTT',
   'TTT',
   'TTT',
   'TTC',
   'TCA',
   'CAT',
   'ATT',
   'TTC',
   'TCT',
   'CTG',
   'TGA',
   'GAC',
   'ACT',
   'CTG',
   'TGC',
   'GCA',
   'CAA',
   'AAC',
   'ACG',
   'CGG',
   'GGG',
   'GGC',
   'GCA',
   'CAA',
   'AAT',
   'ATA',
   'TAT',
   'ATG',
   'TGT',
   'GTC',
   'TCT',
   'CTC',
   'TCT',
   'CTG',
   'TGT',
   'GTG',

In [60]:
# Flattening every line into : (key,1)
mapped_pairs = pairs.map(lambda line : [(pair, 1) for pair in line])
mapped_pairs.take(1)

[[('AG', 1),
  ('GC', 1),
  ('CT', 1),
  ('TT', 1),
  ('TT', 1),
  ('TT', 1),
  ('TC', 1),
  ('CA', 1),
  ('AT', 1),
  ('TT', 1),
  ('TC', 1),
  ('CT', 1),
  ('TG', 1),
  ('GA', 1),
  ('AC', 1),
  ('CT', 1),
  ('TG', 1),
  ('GC', 1),
  ('CA', 1),
  ('AA', 1),
  ('AC', 1),
  ('CG', 1),
  ('GG', 1),
  ('GG', 1),
  ('GC', 1),
  ('CA', 1),
  ('AA', 1),
  ('AT', 1),
  ('TA', 1),
  ('AT', 1),
  ('TG', 1),
  ('GT', 1),
  ('TC', 1),
  ('CT', 1),
  ('TC', 1),
  ('CT', 1),
  ('TG', 1),
  ('GT', 1),
  ('TG', 1),
  ('GT', 1),
  ('TG', 1),
  ('GG', 1),
  ('GA', 1),
  ('AT', 1),
  ('TT', 1),
  ('TA', 1),
  ('AA', 1),
  ('AA', 1),
  ('AA', 1),
  ('AA', 1),
  ('AA', 1),
  ('AA', 1),
  ('AG', 1),
  ('GA', 1),
  ('AG', 1),
  ('GT', 1),
  ('TG', 1),
  ('GT', 1),
  ('TC', 1),
  ('CT', 1),
  ('TG', 1),
  ('GA', 1),
  ('AT', 1),
  ('TA', 1),
  ('AG', 1),
  ('GC', 1),
  ('CA', 1),
  ('AG', 1),
  ('GC', 1),
  ('AGC', 1),
  ('GCT', 1),
  ('CTT', 1),
  ('TTT', 1),
  ('TTT', 1),
  ('TTC', 1),
  ('TCA', 1),
  ('C

---
---
# **Airline Services Tweets Handling**
---
---

In [18]:
# Reading csv file using SparkSession
df = Tweets_sc.read.csv("/content/drive/MyDrive/Colab_Notebooks/tweets.csv", header=True, inferSchema=True)
df.createOrReplaceTempView("df_table")

In [19]:
df.show()

+--------------------+-----------------+----------------------------+--------------------+-------------------------+----------+---------------+--------------------+--------------------+--------------------+
|            tweet_id|airline_sentiment|airline_sentiment_confidence|      negativereason|negativereason_confidence|   airline|           name|                text|       tweet_created|       user_timezone|
+--------------------+-----------------+----------------------------+--------------------+-------------------------+----------+---------------+--------------------+--------------------+--------------------+
|5.67588278875214E...|          neutral|                           1|                NULL|                     NULL|     Delta|    JetBlueNews|@JetBlue's new CE...|2015-02-16 23:36:...|              Sydney|
|5.67590027375702E...|         negative|                           1|          Can't Tell|                   0.6503|     Delta|      nesi_1992|@JetBlue is REALL...|2015-02-

### **1) Which airline company,with low sentiment confidence (< 0.5), has the greatest tweets percentage?**

In [32]:
# Creating table with tweets
query = """
  SELECT airline, COUNT(*) AS total_tweets, SUM(CASE WHEN TRY_CAST(airline_sentiment_confidence AS DOUBLE) < 0.5 THEN 1 ELSE 0 END) AS low_sentiment_tweets
  FROM df_table
  GROUP BY airline
"""

df_low_sentim_tweets = Tweets_sc.sql(query)
df_low_sentim_tweets.createOrReplaceTempView("df_low_sentim_tweets")
df_low_sentim_tweets.show()

+--------------+------------+--------------------+
|       airline|total_tweets|low_sentiment_tweets|
+--------------+------------+--------------------+
|         Delta|        2118|                  50|
|          NULL|           1|                   0|
|Virgin America|         463|                  10|
|        United|        3649|                  57|
|    US Airways|        2767|                  37|
|     Southwest|        2301|                  45|
|      American|        2634|                  25|
+--------------+------------+--------------------+



In [39]:
# Table with tweets percentages
query = """
  SELECT airline, (low_sentiment_tweets / total_tweets)*100 AS tweets_percentage
  FROM df_low_sentim_tweets
  ORDER BY tweets_percentage DESC
  LIMIT 1
"""

df_tweets_percentage = Tweets_sc.sql(query)
df_tweets_percentage.show()

+-------+------------------+
|airline| tweets_percentage|
+-------+------------------+
|  Delta|2.3607176581680833|
+-------+------------------+



### **2) Which airline companies have the worst ratio (negative / total)?**

In [42]:
# Finding total (negative) tweets per company.
query = """
  SELECT airline, COUNT(*) AS total_tweets, SUM(CASE WHEN airline_sentiment = "negative" THEN 1 ELSE 0 END) AS neg_tweets
  FROM df_table
  GROUP BY airline
 """

df_neg_tweets = Tweets_sc.sql(query)
df_neg_tweets.createOrReplaceTempView("df_neg_tweets")
df_neg_tweets.show()

+--------------+------------+----------+
|       airline|total_tweets|neg_tweets|
+--------------+------------+----------+
|         Delta|        2118|       902|
|          NULL|           1|         0|
|Virgin America|         463|       167|
|        United|        3649|      2515|
|    US Airways|        2767|      2152|
|     Southwest|        2301|      1120|
|      American|        2634|      1875|
+--------------+------------+----------+



In [44]:
# Table with the greatest (negative/total) ratio.
query = """
  SELECT airline, (neg_tweets / total_tweets)*100 AS neg_tweets_percentage
  FROM df_neg_tweets
  ORDER BY neg_tweets_percentage DESC
  LIMIT 1
"""

df_neg_tweets_percentage = Tweets_sc.sql(query)
df_neg_tweets_percentage.show()

+----------+---------------------+
|   airline|neg_tweets_percentage|
+----------+---------------------+
|US Airways|    77.77376219732562|
+----------+---------------------+



### **3) Find how many tweets, have greater confidence than their airline company average(for each company).**

In [51]:
# Tweets above average confidence count.
query = """
  WITH AirlineAvgConfidence AS (
    SELECT
      airline,
      AVG(TRY_CAST(airline_sentiment_confidence AS DOUBLE)) AS avg_confidence
    FROM df_table
    GROUP BY airline
  )
  SELECT
    t.airline,
    SUM(CASE WHEN TRY_CAST(t.airline_sentiment_confidence AS DOUBLE) > avg.avg_confidence THEN 1 ELSE 0 END) AS tweets_confidence
  FROM df_table t
  JOIN AirlineAvgConfidence avg
    ON t.airline = avg.airline
  GROUP BY t.airline
  ORDER BY t.airline
"""

df_avg_confidence = Tweets_sc.sql(query)
df_avg_confidence.show()

+--------------+-----------------+
|       airline|tweets_confidence|
+--------------+-----------------+
|      American|             1995|
|         Delta|             1338|
|     Southwest|             1546|
|    US Airways|             2139|
|        United|             2613|
|Virgin America|              301|
+--------------+-----------------+



### **4) Find the five words in the tweets text, that appear most frequently for each airline_sentiment   i) positive,    ii) negative, and  iii) neutral**

In [52]:
# Removing punctuation marks from text.
df = df.withColumn("comment", regexp_replace(col("text"), "[^a-zA-Z0-9 ]", ""))
df = df.drop("text")

# Recreated view with cleaned comments.
df.createOrReplaceTempView("df_table")
df.show()

+--------------------+-----------------+----------------------------+--------------------+-------------------------+----------+---------------+--------------------+--------------------+--------------------+
|            tweet_id|airline_sentiment|airline_sentiment_confidence|      negativereason|negativereason_confidence|   airline|           name|       tweet_created|       user_timezone|             comment|
+--------------------+-----------------+----------------------------+--------------------+-------------------------+----------+---------------+--------------------+--------------------+--------------------+
|5.67588278875214E...|          neutral|                           1|                NULL|                     NULL|     Delta|    JetBlueNews|2015-02-16 23:36:...|              Sydney|JetBlues new CEO ...|
|5.67590027375702E...|         negative|                           1|          Can't Tell|                   0.6503|     Delta|      nesi_1992|2015-02-16 23:43:...|Pacific 

In [53]:
query_Neg = """
    SELECT comment FROM df_table
    WHERE  airline_sentiment == "negative"
"""

query_Pos = """
    SELECT comment FROM df_table
    WHERE  airline_sentiment == "positive"
"""

query_Neu = """
    SELECT comment FROM df_table
    WHERE  airline_sentiment == "neutral"
"""

df_negatives = Tweets_sc.sql(query_Neg)
df_positives = Tweets_sc.sql(query_Pos)
df_neutral = Tweets_sc.sql(query_Neu)

# Creating view of the new tables
df_neutral.createOrReplaceTempView("tab_neutral")
df_positives.createOrReplaceTempView("tab_pos")
df_negatives.createOrReplaceTempView("tab_neg")

In [54]:
# Finding most frequent words for neutral sentiment.
query = """
  SELECT word, COUNT(*) AS frequency
FROM (
    SELECT explode(
        split(
            trim(
                regexp_replace(lower(comment), '[^a-z]+', ' ')
            ),
            '\\s+'
        )
    ) AS word
    FROM tab_neutral
) AS exploded_words
WHERE word != ''
  AND length(word) > 1
GROUP BY word
ORDER BY frequency DESC
LIMIT 5
"""

print("Top 5 most frequent words for Neutral sentiment:")
Tweets_sc.sql(query).show()
df_top5_neu_words = Tweets_sc.sql(query)

Top 5 most frequent words for Neutral sentiment:
+--------------+---------+
|          word|frequency|
+--------------+---------+
|        outhwe|      660|
|        airway|      387|
|             i|       59|
|tinationdragon|       49|
|         enger|       46|
+--------------+---------+



In [55]:
# Finding most frequent words for positive sentiment.
query_pos_words = """
  SELECT word, COUNT(*) AS frequency
FROM (
    SELECT explode(
        split(
            trim(
                regexp_replace(lower(comment), '[^a-z]+', ' ')
            ),
            '\\s+'
        )
    ) AS word
    FROM tab_pos
) AS exploded_words
WHERE word != ''
  AND length(word) > 1
GROUP BY word
ORDER BY frequency DESC
LIMIT 5
"""


print("Top 5 most frequent words for Positive sentiment:")
Tweets_sc.sql(query_pos_words).show()
df_top5_pos_words = Tweets_sc.sql(query_pos_words)

Top 5 most frequent words for Positive sentiment:
+-------------+---------+
|         word|frequency|
+-------------+---------+
|       outhwe|      597|
|       airway|      256|
|       tomer |       85|
|jetblue thank|       70|
|        thank|       64|
+-------------+---------+



In [56]:
# Finding most frequent words for negative sentiment.
query_neg_words = """
  SELECT word, COUNT(*) AS frequency
FROM (
    SELECT explode(
        split(
            trim(
                regexp_replace(lower(comment), '[^a-z]+', ' ')
            ),
            '\\s+'
        )
    ) AS word
    FROM tab_neg
) AS exploded_words
WHERE word != ''
  AND length(word) > 1
GROUP BY word
ORDER BY frequency DESC
LIMIT 5
"""


print("Top 5 most frequent words for Negative sentiment:")
Tweets_sc.sql(query_neg_words).show()
df_top5_neg_words = Tweets_sc.sql(query_neg_words)

Top 5 most frequent words for Negative sentiment:
+------+---------+
|  word|frequency|
+------+---------+
|airway|     2299|
|outhwe|     1198|
|tomer |      443|
|     i|      382|
| tomer|      272|
+------+---------+



### **5) What is the main cause of complaints (negativereason) for each airline? Consider tweets with negativereason_confidence > 0.5.**

In [57]:
# Main cause of complaint for each airline company.
query = """
  SELECT airline, negativereason, count
  FROM (
    SELECT
        airline,
        negativereason,
        count,
        ROW_NUMBER() OVER (
            PARTITION BY airline
            ORDER BY count DESC
        ) AS rn
    FROM (  SELECT airline, negativereason, COUNT(*) AS count FROM df_table
  WHERE negativereason_confidence > 0.5
  GROUP BY airline, negativereason
  ORDER BY airline )
  ) t
  WHERE rn = 1
"""

df_top_neg_reasons = Tweets_sc.sql(query)
Tweets_sc.sql(query).show()

+--------------+--------------------+-----+
|       airline|      negativereason|count|
+--------------+--------------------+-----+
|      American|Customer Service ...|  654|
|         Delta|         Late Flight|  228|
|     Southwest|Customer Service ...|  323|
|    US Airways|Customer Service ...|  698|
|        United|Customer Service ...|  545|
|Virgin America|Customer Service ...|   51|
+--------------+--------------------+-----+

